In [2]:
# yolo method based people counter (demo)

This notebook will demonstrate how to use a pre-trained YOLOv8 model to detect people in single images and count them. The goal if this is 
to produce an annotated image and a numeric count for each input image.

Notes:
* This uses a pre-trained COCO model (`person` class index = 0)
* For training a custom model later, there will be additional instructions below
* Running detection on a CPU is possible but slower. GPU is recommended for speed

SyntaxError: invalid syntax (3581348855.py, line 3)

In [3]:
import sys
print("Python executable:", sys.executable)

# installing needed packages
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install ultralytics opencv-python-headless matplotlib pandas

Python executable: /opt/homebrew/Cellar/jupyterlab/4.4.2/libexec/bin/python


In [2]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path
import os

# show image with matplotlib
def show_image(img, figsize=(10,6), title=None):
    plt.figure(figsize=figsize)
    if img.ndim == 3 and img.shape[2] == 3:
        plt.imshow(img)
    else:
        plt.imshow(img, cmap='gray')
    if title:
        plt.title(title)
    plt.axis('off')
    plt.show()

# draw bounding boxes
def draw_boxes_on_image(img, boxes, labels=None):
    # img in RGB
    img_copy = img.copy()
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box[:4])
        cv2.rectangle(img_copy, (x1, y1), (x2, y2), (0,255,0), 2)
        if labels is not None:
            cv2.putText(img_copy, labels[i], (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
    return img_copy


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/Users/arif/Library/Application Support/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
# load in light pre-trained YOLOv8 variant for quick demo
model = YOLO("yolov8n.pt")
print("Model loaded:", model)

NameError: name 'YOLO' is not defined

In [ ]:
Place 1 or more test images in a folder `data/` inside your project:
- data/
  - sample1.jpg
  - sample2.jpg

Preferably images showing people (no identifiable faces if you will share the repo).  
We'll run detection on every image in `data/` and produce annotated images + a CSV with counts.

In [4]:
image_path = "../data/gym1.jpg" 
assert Path(image_path).exists(), f"Image not found: {image_path}"

# run detection
results = model.predict(source=image_path, imgsz=640, conf=0.25, iou=0.45, max_det=300)
res = results[0] 

# ultralytics result object: res.boxes.xyxy, res.boxes.cls, res.boxes.conf
boxes = res.boxes.xyxy.cpu().numpy() if hasattr(res.boxes, 'xyxy') else []
classes = res.boxes.cls.cpu().numpy().astype(int) if hasattr(res.boxes, 'cls') else []
confs = res.boxes.conf.cpu().numpy() if hasattr(res.boxes, 'conf') else []

# coco class 0 == 'person'. keep only detections where class == 0
person_indices = [i for i, c in enumerate(classes) if c == 0]
person_boxes = [boxes[i] for i in person_indices]
person_confs = [confs[i] for i in person_indices]

count_people = len(person_boxes)
print(f"Detected people (class 0) in image: {count_people}")

# annotate image and show
img_bgr = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
labels = [f"person {i+1} ({person_confs[i]:.2f})" for i in range(len(person_boxes))]
annotated = draw_boxes_on_image(img_rgb, person_boxes, labels)
show_image(annotated, title=f"Detected people: {count_people}")


NameError: name 'Path' is not defined

In [6]:
input_folder = Path("../data")
output_folder = Path("../outputs")
output_folder.mkdir(parents=True, exist_ok=True)

rows = []
for img_file in sorted(input_folder.glob("*.*")):
    if img_file.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
        continue
    results = model.predict(source=str(img_file), imgsz=640, conf=0.25, iou=0.45, max_det=300)
    res = results[0]
    boxes = res.boxes.xyxy.cpu().numpy()
    classes = res.boxes.cls.cpu().numpy().astype(int)
    confs = res.boxes.conf.cpu().numpy()
    # filter person detections
    person_indices = [i for i, c in enumerate(classes) if c == 0]
    person_boxes = [boxes[i] for i in person_indices]
    person_confs = [confs[i] for i in person_indices]
    count_people = len(person_boxes)

    # annotate and save
    img_bgr = cv2.imread(str(img_file))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    labels = [f"person {i+1} ({person_confs[i]:.2f})" for i in range(len(person_boxes))]
    annotated = draw_boxes_on_image(img_rgb, person_boxes, labels)
   
    annotated_bgr = cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR)
    out_path = output_folder / f"annotated_{img_file.name}"
    cv2.imwrite(str(out_path), annotated_bgr)

    rows.append({"image": img_file.name, "count": count_people})

# save csv
df_counts = pd.DataFrame(rows)
csv_path = output_folder / "counts.csv"
df_counts.to_csv(csv_path, index=False)
print("Saved annotated images and counts to", output_folder.resolve())
display(df_counts)


NameError: name 'Path' is not defined